<a href="https://colab.research.google.com/github/marcouras/AI-engineering-fundamentals/blob/main/lezione4/Lezione4_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---

# 🤖 AI Engineering Fundamentals
## Lezione 4 — RAG: Conoscenza Personalizzata

**ITS Novitas 4.0 — Sviluppatore Intelligenza Artificiale**  
Docente: Marco Uras | 📅 Giovedì 28/05/2026

---

### 🎯 Obiettivi
- ✅ Capire la pipeline RAG completa
- ✅ Indicizzare un PDF con ChromaDB
- ✅ Implementare la ricerca semantica
- ✅ Integrare RAG nel chatbot esistente

In [ ]:
# Setup
#!pip install anthropic chromadb pypdf sentence-transformers -q
#from google.colab import userdata
import anthropic, os
from dotenv import load_dotenv  #per usarlo in locale 

#os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
load_dotenv()                              #per usarlo in locale 

client = anthropic.Anthropic()

def chiedi_claude(domanda, system=None, max_tokens=800):
    params = {"model":"claude-haiku-4-5-20251001","max_tokens":max_tokens,
              "messages":[{"role":"user","content":domanda}]}
    if system: params["system"] = system
    return client.messages.create(**params).content[0].text

print("✅ Setup completato!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.5/837.5 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.9/343.9 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6

---
## 1. Crea un documento di test

Per l'esercizio creiamo un documento di testo su WiData. In un progetto reale useresti un PDF vero.

In [2]:
# Creiamo un documento di testo su WiData
documento_widata = """
WiData Srl — Manuale Prodotti IoT

SENSORE XS200 - MONITORAGGIO AMBIENTALE
Il sensore XS200 è progettato per il monitoraggio ambientale in ambienti industriali e urbani.
Misura temperatura (-20°C a +60°C), umidità relativa (0-100%), pressione atmosferica e qualità dell'aria (CO2, PM2.5).
Classificazione IP67: impermeabile e resistente alla polvere. Alimentazione: batteria Li-Ion 3.7V, autonomia 2 anni.
Connettività: LoRaWAN, NB-IoT, WiFi 802.11n. Dimensioni: 85x45x30mm. Peso: 120g.
Certificazioni: CE, FCC, RoHS. Garanzia: 3 anni.

GATEWAY GW500 - CONCENTRATORE DATI
Il gateway GW500 raccoglie dati da fino a 1000 sensori simultaneamente tramite LoRaWAN.
Copertura fino a 15km in aree rurali, 3km in aree urbane. Elaborazione edge computing integrata.
Connessione cloud via Ethernet, WiFi o 4G LTE. Storage locale: 32GB SSD.
Alimentazione: 220V AC o pannello solare (opzionale). Temperatura operativa: -40°C a +70°C.
Certificazioni: CE, IP65. Installazione: palo, tetto o rack.

PIATTAFORMA XPLORE - ANALYTICS
Xplore è la piattaforma cloud di WiData per la visualizzazione e analisi dei dati IoT.
Dashboard personalizzabili con grafici real-time, storico dati fino a 5 anni.
Alerting automatico via email, SMS o webhook quando i valori superano soglie configurabili.
API REST per integrazione con sistemi terzi (ERP, SCADA, BIM).
Machine learning per previsione anomalie e manutenzione predittiva.
Piani: Free (5 sensori), Pro (100 sensori, €49/mese), Enterprise (illimitato, prezzo su richiesta).
SLA: 99.9% uptime garantito nei piani Pro ed Enterprise.

SUPPORTO E ASSISTENZA
Supporto tecnico disponibile lunedì-venerdì 9:00-18:00.
Email: support@widata.cloud | Telefono: +39 079 123456.
Per informazioni commerciali: sales@widata.cloud.
Sede: Via Roma 42, Sassari (SS) 07100, Italia.
Spedizioni in tutta Italia in 3-5 giorni lavorativi.
"""

# Salva su file
with open("manuale_widata.txt", "w", encoding="utf-8") as f:
    f.write(documento_widata)

print(f"✅ Documento creato: {len(documento_widata)} caratteri")

✅ Documento creato: 1846 caratteri


---
## 2. Chunking e Indicizzazione

In [ ]:
def chunka_testo(testo, chunk_size=400, overlap=50):
    """Divide il testo in chunk con overlap."""
    chunks = []
    start = 0
    while start < len(testo):
        end = start + chunk_size
        chunk = testo[start:end]
        if chunk.strip():  # ignora chunk vuoti
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

chunks = chunka_testo(documento_widata)
print(f"📊 Numero di chunk: {len(chunks)}")
print()
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} ({len(chunk)} char) ---")
    print(chunk)
    print()

📊 Numero di chunk: 6

--- Chunk 1 (400 char) ---

WiData Srl — Manuale Prodotti IoT

SENSORE XS200 - MONITORAGGIO AMBIENTALE
Il sensore XS200 è progettato per il monitoraggio ambientale in ambienti industriali e urbani.
Misura temperatura (-20°C a +60°C), umidità relativa (0-100%), pressione atmosferica e qualità dell'aria (CO2, PM2.5).
Classificazione IP67: impermeabile e resistente alla polvere. Alimentazione: batteria Li-Ion 3.7V, autonomia 2

--- Chunk 2 (400 char) ---
. Alimentazione: batteria Li-Ion 3.7V, autonomia 2 anni.
Connettività: LoRaWAN, NB-IoT, WiFi 802.11n. Dimensioni: 85x45x30mm. Peso: 120g.
Certificazioni: CE, FCC, RoHS. Garanzia: 3 anni.

GATEWAY GW500 - CONCENTRATORE DATI
Il gateway GW500 raccoglie dati da fino a 1000 sensori simultaneamente tramite LoRaWAN.
Copertura fino a 15km in aree rurali, 3km in aree urbane. Elaborazione edge computing int

--- Chunk 3 (400 char) ---
km in aree urbane. Elaborazione edge computing integrata.
Connessione cloud via Ethernet, WiF

In [4]:
import chromadb

# Crea il client ChromaDB in memoria
chroma_client = chromadb.Client()

# Crea o recupera la collection
collection = chroma_client.get_or_create_collection(
    name="widata_docs",
    metadata={"hnsw:space": "cosine"}  # usa similarità coseno
)

# Indicizza i chunk
collection.add(
    documents=chunks,
    ids=[f"chunk_{i}" for i in range(len(chunks))]
)

print(f"✅ Indicizzati {collection.count()} chunk in ChromaDB")
print("💡 ChromaDB ha calcolato automaticamente gli embedding per ogni chunk!")

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 32.9MiB/s]


✅ Indicizzati 6 chunk in ChromaDB
💡 ChromaDB ha calcolato automaticamente gli embedding per ogni chunk!


---
## 3. Ricerca Semantica

In [5]:
def cerca(domanda, n_risultati=3):
    """Cerca i chunk più rilevanti per la domanda."""
    risultati = collection.query(
        query_texts=[domanda],
        n_results=n_risultati
    )
    return risultati["documents"][0]

# Test ricerca semantica
domande_test = [
    "Quali sensori supportate per ambienti esterni?",
    "Come posso integrare i dati con il mio sistema ERP?",
    "Qual è il costo del piano professionale?",
]

for domanda in domande_test:
    print(f"\n❓ {domanda}")
    chunks_trovati = cerca(domanda, n_risultati=2)
    for i, chunk in enumerate(chunks_trovati):
        print(f"  📄 Chunk {i+1}: {chunk}")


❓ Quali sensori supportate per ambienti esterni?
  📄 Chunk 1: va.
Piani: Free (5 sensori), Pro (100 sensori, €49/mese), Enterprise (illimitato, prezzo su richiesta).
SLA: 99.9% uptime garantito nei piani Pro ed Enterprise.

SUPPORTO E ASSISTENZA
Supporto tecnico disponibile lunedì-venerdì 9:00-18:00.
Email: support@widata.cloud | Telefono: +39 079 123456.
Per informazioni commerciali: sales@widata.cloud.
Sede: Via Roma 42, Sassari (SS) 07100, Italia.
Spedizi
  📄 Chunk 2: iData per la visualizzazione e analisi dei dati IoT.
Dashboard personalizzabili con grafici real-time, storico dati fino a 5 anni.
Alerting automatico via email, SMS o webhook quando i valori superano soglie configurabili.
API REST per integrazione con sistemi terzi (ERP, SCADA, BIM).
Machine learning per previsione anomalie e manutenzione predittiva.
Piani: Free (5 sensori), Pro (100 sensori, €49

❓ Come posso integrare i dati con il mio sistema ERP?
  📄 Chunk 1: va.
Piani: Free (5 sensori), Pro (100 sensori, €49/mes

---
## 4. RAG Completo — Domanda + Contesto + Risposta

In [6]:
SYSTEM_WIDATA = """
Sei l'assistente virtuale di WiData Srl, azienda IoT e smart cities di Sassari.
Rispondi SOLO basandoti sui documenti forniti nel contesto.
Se la risposta non è nei documenti, dì chiaramente: 'Non ho questa informazione nei miei documenti.'
Non inventare mai informazioni. Sii conciso e preciso.
"""

def chat_rag(domanda, n_chunks=3):
    """Chatbot con RAG: recupera contesto e genera risposta."""
    # 1. Recupera i chunk rilevanti
    chunks_rilevanti = cerca(domanda, n_risultati=n_chunks)
    contesto = "\n\n---\n\n".join(chunks_rilevanti)

    # 2. Costruisci il prompt aumentato
    prompt = f"""Documenti di riferimento:

                  {contesto}

                    ---

                Domanda dell'utente: {domanda}
              """

    # 3. Genera la risposta
    risposta = chiedi_claude(prompt, system=SYSTEM_WIDATA)
    return risposta, chunks_rilevanti

# Test completo
domanda = "Il sensore XS200 funziona in ambienti molto freddi?"
risposta, chunks = chat_rag(domanda)

print(f"❓ {domanda}")
print(f"\n🤖 {risposta}")
print(f"\n📄 Basato su {len(chunks)} chunk")

❓ Il sensore XS200 funziona in ambienti molto freddi?

🤖 Sì, il sensore XS200 è progettato per funzionare in ambienti freddi. Secondo le specifiche tecniche, misura temperature da **-20°C a +60°C**, quindi supporta ambienti molto freddi fino a -20°C.

Inoltre, grazie alla classificazione **IP67** (impermeabile e resistente alla polvere), il sensore è robusto e affidabile anche in condizioni ambientali difficili.

📄 Basato su 3 chunk


In [7]:
# Test con domanda fuori dai documenti
domanda_off = "Quali sono i migliori smartphone del 2025?"
risposta_off, _ = chat_rag(domanda_off)
print(f"❓ {domanda_off}")
print(f"\n🤖 {risposta_off}")
print("\n💡 Il sistema dovrebbe rifiutarsi di rispondere!")

❓ Quali sono i migliori smartphone del 2025?

🤖 Non ho questa informazione nei miei documenti.

I documenti che ho a disposizione riguardano i prodotti e servizi di WiData Srl (sensori IoT, gateway, piattaforma Xplore, piani tariffari e supporto), non smartphone.

Se hai domande su soluzioni IoT e smart cities di WiData, sarò felice di aiutarti! 😊

💡 Il sistema dovrebbe rifiutarsi di rispondere!


---
## ⭐ Esercizi

In [8]:
NOME_STUDENTE = "Stefano"  # ← SCRIVI IL TUO NOME
if NOME_STUDENTE:
    print(f"✅ Notebook di: {NOME_STUDENTE}")
else:
    print("⚠️ Scrivi il tuo nome!")

✅ Notebook di: Stefano


### Esercizio 1 — Indicizza un documento tuo ★☆☆
Crea un documento di testo su un argomento a tua scelta (può essere anche una dispensa del corso, una ricetta, un regolamento). Indicizzalo in ChromaDB e fai 3 domande. I chunk recuperati sono rilevanti?

In [9]:
# ESERCIZIO 1
mio_documento = """
Ferramenta Mastri gestisce un catalogo attivo di oltre quarantacinquemila articoli che coprono l'utensileria meccanica, la viteria speciale, i sistemi di fissaggio e la sicurezza avanzata. Per quanto riguarda la gestione del magazzino e la disponibilità dei prodotti, il nostro sistema informatico aggiorna le giacenze in tempo reale sul portale aziendale. Questo significa che se un articolo risulta disponibile online, è fisicamente presente nei nostri scaffali ed è pronto per il ritiro o per la spedizione immediata. Nel caso in cui un prodotto sia temporaneamente esaurito, i tempi medi di riassortimento dai nostri fornitori partner variano tra i due e i cinque giorni lavorativi, a seconda che si tratti di marchi nazionali o internazionali.

Dal punto di vista tecnico, tutta la nostra viteria e bulloneria segue rigorosamente gli standard internazionali ISO e le normative DIN. Offriamo elementi di fissaggio in diverse classi di resistenza, dall'acciaio zincato standard fino alle leghe ad alta resistenza come la classe 8.8 e 10.9, oltre a un vasto assortimento di acciaio inossidabile nelle varianti A2 e A4, specifiche per resistere alla corrosione in ambienti marini o chimicamente aggressivi. Per i tasselli e i sistemi di ancoraggio pesante, sia meccanici che chimici a base epossidica o vinilestere, forniamo su richiesta le relative certificazioni europee per il calcolo strutturale e la resistenza al fuoco, indispensabili per i direttori dei lavori nei cantieri edili.

Il reparto dedicato alla sicurezza e alla duplicazione offre un servizio tecnico molto specializzato. Siamo in grado di duplicare chiavi standard, chiavi punzonate di sicurezza e chiavi protette da tessera di proprietà, per le quali richiediamo sempre l'esibizione del tesserino originale a tutela della privacy del cliente. Inoltre, il nostro laboratorio programma radiocomandi per cancelli sulla frequenza standard di 433 megahertz e sulla frequenza di 868 megahertz, sia a codice fisso che con tecnologia rolling code. Configuriamo anche cilindri europei con sistemi a chiave maestra o a chiave uguale, che permettono di aprire diversi accessi, come il cancello, il portone e il garage, utilizzando un'unica chiave personalizzata.

Per i professionisti e le aziende dotate di partita IVA abbiamo strutturato un canale di vendita agevolato. I clienti aziendali possono richiedere l'apertura di un conto tecnico che dà diritto a listini prezzi dedicati, alla ricezione di preventivi formali entro tre ore dalla richiesta e alla fatturazione differita a fine mese. Le spedizioni destinate ai cantieri o alle sedi aziendali vengono affidate ai nostri mezzi interni o a corrieri espressi partner, con tempi di consegna che non superano le ventiquattro ore per le merci già presenti in magazzino. I pagamenti accettati presso il punto vendita includono contanti, carte di credito, bancomat e sistemi di pagamento digitali, mentre per gli ordini a distanza e le forniture aziendali concordiamo solitamente il bonifico bancario anticipato o la ricevuta bancaria al ricevimento della fattura.
"""

# Crea una nuova collection
mia_collection = chroma_client.get_or_create_collection(name="mio_doc", metadata={"hnsw:space": "cosine"})

# Chunka e indicizza
chunks = chunka_testo(mio_documento)
mia_collection.add(documents=chunks, ids=[f"chunk_{i}" for i in range(len(chunks))])
print(f"✅ Indicizzati {mia_collection.count()} chunk in ChromaDB")


# Fai 3 domande e stampa i chunk recuperati

def cerca_mio_doc(domanda, collection_to_query, n_risultati=3):
    """Cerca i chunk più rilevanti per la domanda in una specifica collection."""
    risultati = collection_to_query.query(
        query_texts=[domanda],
        n_results=n_risultati
    )
    return risultati["documents"][0]



domande_test = [
    "Avete bulloni ad alta resistenza in classe 10.9?",
    "In quanto tempo consegnate la merce in cantiere?",
    "Cosa serve per duplicare una chiave con tessera di proprietà?"
]

for domanda in domande_test:
    print(f"\n❓ {domanda}")
    chunks_trovati = cerca_mio_doc(domanda, mia_collection, n_risultati=2)
    for i, chunk in enumerate(chunks_trovati):
        print(f"  📄 Chunk {i+1}: {chunk}")











✅ Indicizzati 9 chunk in ChromaDB

❓ Avete bulloni ad alta resistenza in classe 10.9?
  📄 Chunk 1: e si tratti di marchi nazionali o internazionali.

Dal punto di vista tecnico, tutta la nostra viteria e bulloneria segue rigorosamente gli standard internazionali ISO e le normative DIN. Offriamo elementi di fissaggio in diverse classi di resistenza, dall'acciaio zincato standard fino alle leghe ad alta resistenza come la classe 8.8 e 10.9, oltre a un vasto assortimento di acciaio inossidabile ne
  📄 Chunk 2: a un vasto assortimento di acciaio inossidabile nelle varianti A2 e A4, specifiche per resistere alla corrosione in ambienti marini o chimicamente aggressivi. Per i tasselli e i sistemi di ancoraggio pesante, sia meccanici che chimici a base epossidica o vinilestere, forniamo su richiesta le relative certificazioni europee per il calcolo strutturale e la resistenza al fuoco, indispensabili per i d

❓ In quanto tempo consegnate la merce in cantiere?
  📄 Chunk 1: endale. Questo signif

### Esercizio 2 — Sperimenta con il chunking ★★☆
Prova a indicizzare lo stesso documento con chunk_size=200, 400 e 800. Per la stessa domanda, i chunk recuperati sono diversi? Quale dimensione dà risultati migliori?

In [10]:
# ESERCIZIO 2
domanda_test = "Avete bulloni ad alta resistenza in classe 10.9?"  # ← modifica

for chunk_size in [200, 400, 800]:
    print(f"\n{'='*50}")
    print(f"chunk_size = {chunk_size}")
    print('='*50)

    # TODO: crea collection, chunka con dimensione diversa, indicizza, cerca

    # Crea una nuova collection
    nuova_collection = chroma_client.get_or_create_collection(name=f"nuovo_doc_{chunk_size}", metadata={"hnsw:space": "cosine"})

    # Chunka e indicizza
    overlap_dinamico=int(chunk_size/4)
    chunks = chunka_testo(mio_documento, chunk_size=chunk_size, overlap=overlap_dinamico)
    nuova_collection.add(documents=chunks, ids=[f"chunk_{i}" for i in range(len(chunks))])
    print(f"✅ Indicizzati {nuova_collection.count()} chunk nella collection in ChromaDB")


    print(f"\n❓ {domanda_test}")
    chunks_trovati = cerca_mio_doc(domanda_test, nuova_collection, n_risultati=1)

    print(f"\n🎯 Risultato della ricerca per chunk_size {chunk_size}:")
    testo = chunks_trovati[0].replace("\n", " ").strip()

    print(f"\"{testo}...\"")



# Commento: quale chunk_size ha dato i risultati migliori?
# Risposta:



chunk_size = 200
✅ Indicizzati 21 chunk nella collection in ChromaDB

❓ Avete bulloni ad alta resistenza in classe 10.9?

🎯 Risultato della ricerca per chunk_size 200:
"menti di fissaggio in diverse classi di resistenza, dall'acciaio zincato standard fino alle leghe ad alta resistenza come la classe 8.8 e 10.9, oltre a un vasto assortimento di acciaio inossidabile ne..."

chunk_size = 400
✅ Indicizzati 11 chunk nella collection in ChromaDB

❓ Avete bulloni ad alta resistenza in classe 10.9?

🎯 Risultato della ricerca per chunk_size 400:
"sortimento dai nostri fornitori partner variano tra i due e i cinque giorni lavorativi, a seconda che si tratti di marchi nazionali o internazionali.  Dal punto di vista tecnico, tutta la nostra viteria e bulloneria segue rigorosamente gli standard internazionali ISO e le normative DIN. Offriamo elementi di fissaggio in diverse classi di resistenza, dall'acciaio zincato standard fino alle leghe ad..."

chunk_size = 800
✅ Indicizzati 6 chunk nella coll

### Esercizio 3 — RAG + storia conversazione ★★☆
Integra RAG nella funzione `chat()` della Lezione 3 (quella con la history). Ogni risposta deve usare sia il contesto RAG che la storia della conversazione.

In [12]:
# ESERCIZIO 3



SYSTEM_WIDATA = """
                  Sei l'assistente virtuale di WiData Srl, azienda IoT e smart cities di Sassari.
                  Rispondi SOLO basandoti sui documenti forniti nel contesto.
                  Se la risposta non è nei documenti, dì chiaramente: 'Non ho questa informazione nei miei documenti.'
                  Non inventare mai informazioni. Sii conciso e preciso.
                """

history = []

def chat_rag_con_storia(domanda, system, n_chunks=2):
    """Chatbot con RAG + conversazione multi-turno."""
    # TODO:
    # 1. Recupera chunk rilevanti con cerca()
    # 2. Costruisci il messaggio con contesto + domanda
    # 3. Aggiungi alla history
    # 4. Chiama l'API con tutta la history
    # 5. Aggiungi la risposta alla history
    # 6. Restituisci la risposta

    chunks_rilevanti = cerca(domanda, n_risultati=n_chunks)
    contesto = "\n\n---\n\n".join(chunks_rilevanti)
    prompt = f"""Documenti di riferimento:
                      {contesto}
                          ---
                 Domanda dell'utente: {domanda}"""


    history.append({"role": "user", "content": prompt})

    params = {
        "model": "claude-haiku-4-5-20251001",
        "max_tokens": 500,
        "messages": history
    }
    if system:
        params["system"] = system

    risposta = client.messages.create(**params)
    testo = risposta.content[0].text


    history.append({"role": "assistant", "content": testo})

    return testo





# Test: domande collegate che richiedono sia RAG che memoria
domanda1 = "Parlami del sensore XS200"
risposta1 = chat_rag_con_storia(domanda1, SYSTEM_WIDATA)

domanda2 = "Qual è la sua autonomia?"
risposta2 = chat_rag_con_storia(domanda2, SYSTEM_WIDATA)

domanda3 = "Costa molto?"
risposta3 = chat_rag_con_storia(domanda3, SYSTEM_WIDATA)

print("Domanda1: ", domanda1)
print("Risposta1: ", risposta1)
print("\n", "="*100, "\n")
print("Domanda2: ", domanda2)
print("Risposta2: ", risposta2)
print("\n", "="*100, "\n")
print("Domanda3: ", domanda3)
print("Risposta3: ", risposta3)


Domanda1:  Parlami del sensore XS200
Risposta1:  # Sensore XS200 - Monitoraggio Ambientale

Il **sensore XS200** è progettato per il monitoraggio ambientale in ambienti industriali e urbani.

## Caratteristiche tecniche:

**Parametri misurati:**
- Temperatura: -20°C a +60°C
- Umidità relativa: 0-100%
- Pressione atmosferica
- Qualità dell'aria (CO2, PM2.5)

**Resistenza:**
- Classificazione IP67: impermeabile e resistente alla polvere

**Alimentazione:**
- Batteria Li-Ion 3.7V

Per informazioni più dettagliate sulla batteria e altre specifiche, ti consiglio di contattare il nostro team di supporto a **support@widata.cloud** o al **+39 079 123456** (lunedì-venerdì 9:00-18:00).


Domanda2:  Qual è la sua autonomia?
Risposta2:  In base ai documenti forniti, posso dirvi che il **sensore XS200** è alimentato da una **batteria Li-Ion 3.7V**, ma purtroppo **il documento non specifica l'autonomia esatta in ore o giorni**.

Per informazioni complete sull'autonomia della batteria, ti consiglio d

### Esercizio 4 — Chatbot RAG WiData completo ★★★ (Deliverable!)

Costruisci il chatbot completo con:
- RAG sul documento WiData
- Conversation history (sliding window)
- Streaming
- System prompt WiData con istruzione anti-hallucination
- Loop interattivo con `input()`
- Stampa i chunk usati per ogni risposta (per debug)

In [ ]:
# ESERCIZIO 4 — Chatbot RAG completo (DELIVERABLE)
import chromadb, json, os
#from google.colab import userdata
import anthropic
import copy

#os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
client = anthropic.Anthropic()


documento_widata = """
WiData Srl — Manuale Prodotti IoT

SENSORE XS200 - MONITORAGGIO AMBIENTALE
Il sensore XS200 è progettato per il monitoraggio ambientale in ambienti industriali e urbani.
Misura temperatura (-20°C a +60°C), umidità relativa (0-100%), pressione atmosferica e qualità dell'aria (CO2, PM2.5).
Classificazione IP67: impermeabile e resistente alla polvere. Alimentazione: batteria Li-Ion 3.7V, autonomia 2 anni.
Connettività: LoRaWAN, NB-IoT, WiFi 802.11n. Dimensioni: 85x45x30mm. Peso: 120g.
Certificazioni: CE, FCC, RoHS. Garanzia: 3 anni.

GATEWAY GW500 - CONCENTRATORE DATI
Il gateway GW500 raccoglie dati da fino a 1000 sensori simultaneamente tramite LoRaWAN.
Copertura fino a 15km in aree rurali, 3km in aree urbane. Elaborazione edge computing integrata.
Connessione cloud via Ethernet, WiFi o 4G LTE. Storage locale: 32GB SSD.
Alimentazione: 220V AC o pannello solare (opzionale). Temperatura operativa: -40°C a +70°C.
Certificazioni: CE, IP65. Installazione: palo, tetto o rack.

PIATTAFORMA XPLORE - ANALYTICS
Xplore è la piattaforma cloud di WiData per la visualizzazione e analisi dei dati IoT.
Dashboard personalizzabili con grafici real-time, storico dati fino a 5 anni.
Alerting automatico via email, SMS o webhook quando i valori superano soglie configurabili.
API REST per integrazione con sistemi terzi (ERP, SCADA, BIM).
Machine learning per previsione anomalie e manutenzione predittiva.
Piani: Free (5 sensori), Pro (100 sensori, €49/mese), Enterprise (illimitato, prezzo su richiesta).
SLA: 99.9% uptime garantito nei piani Pro ed Enterprise.

SUPPORTO E ASSISTENZA
Supporto tecnico disponibile lunedì-venerdì 9:00-18:00.
Email: support@widata.cloud | Telefono: +39 079 123456.
Per informazioni commerciali: sales@widata.cloud.
Sede: Via Roma 42, Sassari (SS) 07100, Italia.
Spedizioni in tutta Italia in 3-5 giorni lavorativi.
"""



SYSTEM = """
            Sei l'assistente virtuale di WiData Srl, azienda IoT e smart cities di Sassari.
            Rispondi SOLO basandoti sui documenti forniti nel contesto.
            Se la risposta non è nei documenti, dì chiaramente: 'Non ho questa informazione nei miei documenti.'
            Non inventare mai informazioni. Sii conciso e preciso.
        """



MAX_MESSAGGI = 10





def setup_rag(testo):
    """Indicizza il documento e restituisce la collection."""
    nome_collection = "info_widata"
    collection_widata = chroma_client.get_or_create_collection(name=nome_collection, metadata={"hnsw:space": "cosine"})
    chunks = chunka_testo(testo)
    collection_widata.add(documents=chunks, ids=[f"chunk_{i}" for i in range(len(chunks))])
    print(f"Sono stati indicizzati {collection_widata.count()} chunk.")
    return collection_widata




def cerca(domanda, collection, n=2):
    risultati = collection.query(
        query_texts=[domanda],
        n_results=n
    )
    return risultati["documents"][0]




def chat_completo(domanda, history, collection):
    """Chatbot con RAG + storia + streaming."""
    chunks_rilevanti = cerca(domanda, collection)
    contesto = "\n---\n".join(chunks_rilevanti)

    prompt = f"""Documenti di riferimento:
                        {contesto}
                            ---
                 Domanda dell'utente: {domanda}"""

    messaggi_da_inviare = copy.deepcopy(history[-MAX_MESSAGGI:])
    messaggi_da_inviare.append({"role": "user", "content": prompt})

    print("\n🤖 Assistente: ", end="", flush=True)

    risposta = ""
    with client.messages.stream(
        model="claude-haiku-4-5-20251001",
        max_tokens=500,
        system=SYSTEM,
        messages=messaggi_da_inviare
    ) as stream:
        for text in stream.text_stream:
            print(text, end="", flush=True)
            risposta += text
    print("\n")

    history.append({"role": "user", "content": domanda})
    history.append({"role": "assistant", "content": risposta})





def main():
    collection = setup_rag(documento_widata)
    history = []
    print("🤖 Chatbot WiData RAG avviato. Digita 'esci' per uscire.\n")

    while True:
        utente = input("Tu: ")
        if utente.lower() == "esci":
            print("👋 Arrivederci!")
            break
        chat_completo(utente, history, collection)

main()  # Decommentare per eseguire

Sono stati indicizzati 6 chunk.
🤖 Chatbot WiData RAG avviato. Digita 'esci' per uscire.

Tu: dimmi le specifiche del sensore XS200

🤖 Assistente: # Specifiche del Sensore XS200

Il sensore **XS200** è progettato per il **monitoraggio ambientale** in ambienti industriali e urbani.

## Parametri misurati:
- **Temperatura**: -20°C a +60°C
- **Umidità relativa**: 0-100%
- **Pressione atmosferica**
- **Qualità dell'aria**: CO2 e PM2.5

## Caratteristiche tecniche:
- **Classificazione IP67**: impermeabile e resistente alla polvere
- **Alimentazione**: Batteria Li-Ion 3.7V

*Nota: La documentazione disponibile non contiene l'informazione completa sull'autonomia della batteria (il testo risulta incompleto).*

Tu: come posso avere assistenza?

🤖 Assistente: # Come Avere Assistenza

WiData Srl mette a disposizione i seguenti canali di supporto:

## Supporto Tecnico
- **Orari**: Lunedì-Venerdì, 9:00-18:00
- **Email**: support@widata.cloud
- **Telefono**: +39 079 123456

## Informazioni Commercial

---
## 📤 Consegna

1. Completa tutti gli esercizi
2. Scarica: `File → Scarica → .ipynb`
3. Rinomina: `Lezione4_TUONOME.ipynb`
4. Carica su GitHub in `lezione4/`

```bash
git add lezione4/
git commit -m "Lezione 4 completata"
git push
```

---
### 📖 Per la prossima lezione (Giovedì 04/06)
Leggi **Huyen Cap. 6 — sezione Agents**

---
*ITS Novitas 4.0 — AI Engineering Fundamentals | Marco Uras*